In [13]:
from pathlib import Path
import sys

# Try to auto-detect the repo root by walking up until we see 01_Data & 03_Analysis
def find_repo_root(start: Path = Path.cwd()) -> Path:
    cur = start.resolve()
    for _ in range(7):  # climb a few levels
        if (cur / "01_Data").exists() and (cur / "03_Analysis").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not locate repo root with 01_Data and 03_Analysis.")

BASE = find_repo_root()
DATA = BASE / "01_Data" / "processed"
FIGS = BASE / "03_Analysis" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# so we can import forecast_utils (which lives in 03_Analysis)
sys.path.append(str((BASE / "03_Analysis").resolve()))

print("BASE:", BASE)
print("DATA exists:", DATA.exists())
print("Expect monthly_revenue.csv at:", DATA / "monthly_revenue.csv")


BASE: C:\Users\huzei\OneDrive\Documents\GitHub\Ecommerce-Finance-Analytics-Portfolio
DATA exists: True
Expect monthly_revenue.csv at: C:\Users\huzei\OneDrive\Documents\GitHub\Ecommerce-Finance-Analytics-Portfolio\01_Data\processed\monthly_revenue.csv


In [14]:
from pathlib import Path
csv_path = Path(BASE) / "01_Data" / "processed" / "monthly_revenue.csv"
print("Exists?", csv_path.exists(), "→", csv_path)


Exists? True → C:\Users\huzei\OneDrive\Documents\GitHub\Ecommerce-Finance-Analytics-Portfolio\01_Data\processed\monthly_revenue.csv


In [21]:
from pathlib import Path
import pandas as pd
from forecast_utils import (
    load_monthly_revenue, fit_sarimax, forecast_sarimax,
    build_forecast_frame, plot_forecast
)

# Detect base path automatically (works for both notebook & script)
try:
    BASE = Path(__file__).resolve().parents[1]  # when running as .py file
except NameError:
    # fallback for notebooks
    BASE = Path.cwd().parents[0]

DATA = BASE / "01_Data" / "processed"
FIGS = BASE / "03_Analysis" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("DATA exists:", DATA.exists())


BASE: c:\Users\huzei\OneDrive\Documents\GitHub\Ecommerce-Finance-Analytics-Portfolio
DATA exists: True


In [22]:
monthly = load_monthly_revenue(DATA / "monthly_revenue.csv")
monthly["y"] = monthly["LineAmount"].astype(float)

model = fit_sarimax(monthly["y"], seasonal_periods=12)
yhat, conf = forecast_sarimax(model, periods=12)
fc = build_forecast_frame(monthly["ds"].max(), yhat, conf)

plot_forecast(
    monthly[["ds","y"]],
    fc,
    "Revenue Forecast (next 12 months)",
    FIGS / "revenue_forecast_12m.png"
)

print("✅ Saved forecast chart:", FIGS / "revenue_forecast_12m.png")


✅ Saved forecast chart: c:\Users\huzei\OneDrive\Documents\GitHub\Ecommerce-Finance-Analytics-Portfolio\03_Analysis\figures\revenue_forecast_12m.png
